Langchain agent with Tavily web search API

In [ ]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
import os
import truststore
from langchain_openai import ChatOpenAI
truststore.inject_into_ssl()

# Set Tavily API key
os.environ["TAVILY_API_KEY"] = "your_tavily_api_key"  # replace with your Tavily API key


llm = ChatOpenAI(
    # model="google/gemma-4-31b-it:free",   # here you can set the model
    model="gpt-4o-mini",   # here you can set the model
    temperature=0,
    api_key="your_openrouter_api_key",  # replace with your OpenRouter API key
    base_url="https://openrouter.ai/api/v1",
    extra_body={"reasoning": {"enabled": True}}  # optional
)
# Initialize Tavily Search Tool
tavily_search_tool = TavilySearch()
tools = [tavily_search_tool]
agent = create_agent(llm, tools)

user_input = "What nation hosted the Euro 2024?"

# for step in agent.stream(
#     {"messages": user_input},
#     stream_mode="values",
# ):
#     step["messages"][-1].pretty_print()
final_result = agent.invoke({"messages": user_input})
final_result["messages"][-1].pretty_print()

================================== Ai Message ==================================

Germany is the host nation for Euro 2024. The tournament is scheduled to take place from June 14 to July 14, 2024. This will be the first time Germany has hosted the UEFA European Championship since reunification, although West Germany previously hosted the tournament in 1988.


Langchain Agent with user prompt and Travily search API

In [ ]:
from langchain_tavily import TavilySearch
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
import os
import truststore
truststore.inject_into_ssl()

# Set Tavily API key
os.environ["TAVILY_API_KEY"] = "your_tavily_api_key"  # replace with your Tavily API key

llm = ChatOpenAI(
    # model="google/gemma-4-31b-it:free",   # here you can set the model
    model="gpt-4o-mini",   # here you can set the model
    temperature=0,
    api_key="your_openrouter_api_key",  # replace with your OpenRouter API key
    base_url="https://openrouter.ai/api/v1",
    extra_body={"reasoning": {"enabled": True}}  # optional
)

# Initialize Tavily Search Tool
tavily_search_tool = TavilySearch(max_results=5,topic="general",)

agent = create_agent(llm, [tavily_search_tool])

Address = "Palisades Charter High School, 15777 Bowdoin St, Pacific Palisades, CA 90272, USA"
Incident_Date = "17/01/2025"
user_input = f"""
user: I want to report a security incident that occurred at {Address} on {Incident_Date}. The incident involved unauthorized access to the school's computer systems, resulting in the compromise of sensitive student and staff data. Please provide guidance on how to proceed with reporting this incident to the appropriate authorities and any immediate steps we should take to mitigate the impact.
"""


for step in agent.stream({"messages": user_input},stream_mode="values",):
    step["messages"][-1].pretty_print()

Building Agent from Scratch

In [ ]:
import os
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableLambda
from langchain_community.tools.tavily_search.tool import TavilySearchResults
from langgraph.graph import StateGraph, END

# -- SETUP --
os.environ["TAVILY_API_KEY"] = "your_tavily_api_key"  # replace with your Tavily API key

llm = ChatOpenAI(
    # model="google/gemma-4-31b-it:free",   # here you can set the model
    model="gpt-4o-mini",   # here you can set the model
    temperature=0,
    api_key="your_openrouter_api_key",  # replace with your OpenRouter API key
    base_url="https://openrouter.ai/api/v1",
    extra_body={"reasoning": {"enabled": True}}  # optional
)

# -- TOOL (unused for now) --
tools = [TavilySearchResults(k=2)]

# -- STATE SCHEMA --
class AgentState(TypedDict):
    messages: list
    input: str
    intermediate_steps: list
    output: str # Include output field in state

# -- AGENT STEP --
def call_agent(state: AgentState) -> AgentState:
    user_input = state["input"]
    input_msg = HumanMessage(content=user_input)
    result = llm.invoke([input_msg])
    return {
    "messages": state.get("messages", []) + [input_msg, result],
    "input": user_input,
    "intermediate_steps": [],
    "output": result.content # explicitly add output
    }

# -- BUILD THE GRAPH --
workflow = StateGraph(AgentState)
workflow.add_node("agent", RunnableLambda(call_agent))
workflow.set_entry_point("agent")
workflow.add_edge("agent", END) # no need for separate finish node now

# -- RUN THE GRAPH --
graph_executor = workflow.compile()


result = graph_executor.invoke({"input": "what is Agentic AI?", "messages": [], "intermediate_steps": [], "output": "" })

print(result["output"])

C:\Users\Aksha\AppData\Local\Temp\ipykernel_18068\735518416.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search.tool import TavilySearchResults
C:\Users\Aksha\AppData\Local\Temp\ipykernel_18068\735518416.py:22: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tools = [TavilySearchResults(k=2)]


Agentic AI refers to artificial intelligence systems that possess a degree of autonomy and can act independently to achieve specific goals or tasks. Unlike traditional AI, which may operate primarily as tools or assistants under human control, agentic AI can make decisions, learn from its environment, and adapt its behavior based on its experiences.

Key characteristics of agentic AI include:

1. **Autonomy**: The ability to operate independently without constant human intervention.
2. **Goal-directed behavior**: The capacity to pursue specific objectives or tasks.
3. **Learning and adaptation**: The ability to learn from experiences and improve performance over time.
4. **Decision-making**: The capability to make choices based on available information and predefined criteria.

Agentic AI can be applied in various domains, including robotics, autonomous vehicles, and complex systems management, where it can enhance efficiency and effectiveness by taking initiative and responding to dyn

Building Agent from Scratch with user prompt

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage, HumanMessage
from langchain_community.tools.tavily_search.tool import TavilySearchResults
from langchain_core.runnables import RunnableConfig
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated
# from langgraph.graph.schema import add_messages
from typing_extensions import Doc

# ---- DEFINE STATE ----
class AgentState(TypedDict):
    input: Annotated[str, Doc("User input")]
    messages: Annotated[list, Doc("Message history")]
    intermediate_steps: Annotated[list, Doc("Search or tool results")]
    search: Annotated[str, Doc("Tavily Search Result")]
    output: Annotated[str, Doc("Final agent output")] 
os.environ["TAVILY_API_KEY"] = "your_tavily_api_key"  # replace with your Tavily API key
# ---- LLM SETUP ----
# llm = AzureChatOpenAI(deployment_name="your-deployment", model_name="gpt-4", temperature=0)
llm = ChatOpenAI(
    # model="google/gemma-4-31b-it:free",   # here you can set the model
    model="gpt-4o-mini",   # here you can set the model
    temperature=0,
    api_key="your_openrouter_api_key",  # replace with your OpenRouter API key
    base_url="https://openrouter.ai/api/v1",
    extra_body={"reasoning": {"enabled": True}}  # optional
)
# ---- TOOL SETUP ----
tavily_tool = TavilySearchResults(k=3)

# ---- SEARCH NODE ----
def run_search(state):
    query = state["input"]
    result = tavily_tool.invoke(query)
    return {"search": result}

# ---- AGENT NODE ----
# def call_agent(state):
#     prompt = state["input"]
#     search = state.get("search", "")
#     messages = [
#         HumanMessage(content=f"{search}\n\n{prompt}")
#     ]
#     response = llm.invoke(messages)
#     return {
#         "messages": add_messages(state, [messages[-1], response]),
#         "output": response.content
#     }
def call_agent(state):
    prompt = state["input"]
    search = state.get("search", "")
    messages = [HumanMessage(content=f"{search}\n\n{prompt}")]
    response = llm.invoke(messages)
    # return {
    #     "messages": state["messages"] + [messages[-1], response],
    #     "output": response.content
    # }
    return {
    "messages": messages,
    "intermediate_steps": ("intermediate_steps"),
    "search": search,
    "output": response.content # Return output explicitly
}

# ---- GRAPH BUILD ----
workflow = StateGraph(AgentState)
workflow.add_node("search_tool", RunnableLambda(run_search))
workflow.add_node("agent", RunnableLambda(call_agent))

workflow.set_entry_point("search_tool")
# workflow.set_finish_point("search_and_respond") 
workflow.add_edge("search_tool", "agent")
workflow.add_edge("agent", END)

graph_executor = workflow.compile()

Address = "Pasadena jewish temple, 1434 N Altadena Dr, Pasadena, CA 91107, USA"
#Address = "Palisades Charter High School, 15777 Bowdoin St, Pacific Palisades, CA 90272, USA"
Incident_Date = "10/01/2025"
prompt = f"""
user: I want to report a security incident that occurred at {Address} on {Incident_Date}. The incident involved unauthorized access to the temple's computer systems, resulting in the compromise of sensitive member and staff data. Please provide guidance on how to proceed with reporting this incident to the appropriate authorities and any immediate steps we should take to mitigate the impact.
"""

# result = graph_executor.invoke({
#     "input": prompt,
#     "messages": [],
#     "intermediate_steps": [],
#     "search": ""
# })
result = graph_executor.invoke({
    "input": prompt,
    "messages": [],
    "search": ""
})

print(result["output"])

Deepagents with function calling

In [ ]:
from deepagents import create_deep_agent
import os
os.environ["OPENROUTER_API_KEY"] = "your_openrouter_api_key"  # replace with your OpenRouter API key

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_deep_agent(
    model="openrouter:z-ai/glm-5.2",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='dee647c0-0d18-4ddf-a1b7-6490dcb19c8c'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'z-ai/glm-5.2', 'id': 'gen-1786268731-1ivtYI7FQQWtxfU6ZWGp', 'created': 1786268731, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 6.156e-05, 'cost_details': {'upstream_inference_completions_cost': 2.86e-06, 'upstream_inference_prompt_cost': 5.87e-05, 'upstream_inference_cost': 6.156e-05}}, id='lc_run--019fe5e9-b9d4-7642-a5a4-394e028ae130-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call_06bfda43b6b74c189aedd3cf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2345, 'output_tokens': 13, 'total_tokens': 2358, 'input_token_details': {'cache_read': 1850, 'cache_creation': 0}, 'output_token_details': {'reasoning': 14}}),
  Too

Deepagent with tool calling

In [ ]:
from deepagents import create_deep_agent
from langchain_tavily import TavilySearch

# def get_weather(city: str) -> str:
#     """Get weather for a given city."""
#     return f"It's always sunny in {city}!"

# Set Tavily API key
os.environ["TAVILY_API_KEY"] = "your_tavily_api_key"  # replace with your Tavily API key

# Initialize Tavily search
tavily_search = TavilySearch()

agent = create_deep_agent(
    model="openrouter:z-ai/glm-5.2",
    tools=[tavily_search],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "Write a research report on the impact of climate change on coastal cities."}]}
)

{'messages': [HumanMessage(content='Write a research report on the impact of climate change on coastal cities.', additional_kwargs={}, response_metadata={}, id='0ee29adb-9d74-4ce5-af10-dd9669814478'),
  AIMessage(content="I'll research this topic thoroughly before writing the report. Let me gather current information from multiple angles.", additional_kwargs={'reasoning_content': 'The user wants a research report on the impact of climate change on coastal cities. I should gather current, well-sourced information to write a comprehensive report. Let me search for relevant information using multiple searches in parallel.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants a research report on the impact of climate change on coastal cities. I should gather current, well-sourced information to write a comprehensive report. Let me search for relevant information using multiple searches in parallel.'}]}, response_metadata={'model_name':